# Test GLONER (GLiNER + LoRA)

Testing the new GLONER system for model initialization and training.

In [ ]:
import sys
sys.path.append('../src')

import os
import torch
from models.gloner import GLONER
from config.lora_defaults import DEFAULT_GLINER_MODEL, DEFAULT_LORA_CONFIG, DEFAULT_MAX_LENGTH
from utils.logging import get_logger
from data.loader import load_mit_dataset
from evaluation.eval import evaluate_gloner
logger = get_logger("GLONERTest")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Check CUDA
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs visible: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    device = "cuda"
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    device = "cpu"
    print("CUDA not available, using CPU")

/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA version: 12.8
Number of GPUs visible: 1
Current GPU: 0
GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Memory: 5.7 GB


In [ ]:


# Show config
print(f"\nDefault GLiNER model: {DEFAULT_GLINER_MODEL}")
print(f"Default GLiNER model max length: {DEFAULT_MAX_LENGTH}")
print("\nDefault LoRA config:")
for key, value in DEFAULT_LORA_CONFIG.items():
    if key == 'target_modules':
        print(f"  {key}: {len(value)} modules")
    else:
        print(f"  {key}: {value}")

# Load test data
print("\n=== Loading Test Data ===")
test_data, entity_types = load_mit_dataset("../data/mit-movie/test.json", "../data/mit-movie/labels.json")
print(f"Loaded {len(test_data)} test examples")
print(f"Entity types: {entity_types}")
print(f"Sample example: {test_data[0]}")

# Test 1: Create trainable GLONER (for training)
print("\n=== Test 1: Creating Trainable GLONER ===")
gloner_train = GLONER.for_training(logger=logger)
print("✓ Created trainable GLONER")
counts = gloner_train.get_param_counts()
print(f"  Total params: {counts['total']:,}")
print(f"  Trainable params: {counts['trainable']:,} ({counts['percentage']:.1f}%)")



predictions = gloner_train.predict(
    data=test_data,
    entity_types=entity_types,
    threshold=0.5,
    batch_size=8,
    device=device
)

print(device)
gloner_train.to(device)
with torch.no_grad():
    gliner_results=gloner_train.evaluate(test_data, entity_types, batch_size=8, threshold=0.5 )

    


In [ ]:
gloner_results=evaluate_gloner(predictions,test_data,entity_types,has_ground_truth=True)
print(f"Evaluation Results: {gloner_results['overall_metrics']['f1_pct']}")
